# 01 — Setup & pilot data (Colab T4)

Clone **musicinstruct**, install the LLM stack, authenticate Hugging Face, generate dataset (synthetic pilot or **real MidiCaps/Lakh v0.2**), validate, and self-test the grader.

Run **Sync to Drive** at the end so notebook **02** can reuse the dataset in a new session.

**Runtime:** GPU → T4 (or better).

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
RUN_ID = "pilot_v1"  # @param {type:"string"}  reuse in notebooks 02/03

DATASET_MODE = "pilot"  # @param ["pilot", "real_v0.2"]  synthetic vs MidiCaps/Lakh seeds
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_ROOT}/data/pilot" if DATASET_MODE == "pilot" else f"{DRIVE_ROOT}/data/v0.2"
LOCAL_DATA = f"{MUSICINSTRUCT}/data/pilot" if DATASET_MODE == "pilot" else f"{MUSICINSTRUCT}/data/v0.2"
MANIFEST_NAME = "pilot.jsonl" if DATASET_MODE == "pilot" else "manifest.jsonl"
TARGET_ITEMS = 300 if DATASET_MODE == "pilot" else 1000  # @param {type:"integer"}
FORCE_REGENERATE = False  # @param {type:"boolean"}  overwrite local + Drive data

# Real v0.2 only — MidiCaps tarball extract or Lakh tree on Drive
DRIVE_MIDICAPS = f"{DRIVE_ROOT}/midicaps"
LAKH_ROOT = f"{WORK}/data/midicaps/lmd_full"  # @param {type:"string"}
SEED_LIMIT = 200  # @param {type:"integer"}  max composition seeds (real mode)
MIDICAPS_SCAN_LIMIT = 5000  # @param {type:"integer"}

In [ ]:
# @title Mount Google Drive (optional)
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DATA).parent.mkdir(parents=True, exist_ok=True)
    print("Drive data path:", DRIVE_DATA)
else:
    print("Drive disabled — data stays on this VM only.")

In [ ]:
# @title Hugging Face token (Llama 3.2 license required)
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# @title Clone repo & install
import shutil
from pathlib import Path

if Path(MUSICINSTRUCT).is_dir():
    shutil.rmtree(MUSICINSTRUCT)
!git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}
%cd {MUSICINSTRUCT}
!pip install -q -U "transformers>=4.44" accelerate huggingface_hub datasets
!pip install -q -e ".[llm,stress]"

In [ ]:
# @title GPU check
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("Warning: no GPU — plan executor will be slow. Enable T4 in Runtime settings.")

In [ ]:
# @title Restore data from Drive (or generate)
import shutil
from pathlib import Path

local_data = Path(LOCAL_DATA)
manifest = local_data / MANIFEST_NAME
drive_manifest = Path(DRIVE_DATA) / MANIFEST_NAME if USE_DRIVE else None

if FORCE_REGENERATE and local_data.exists():
    shutil.rmtree(local_data)
elif USE_DRIVE and drive_manifest and drive_manifest.is_file() and not manifest.is_file():
    shutil.copytree(DRIVE_DATA, local_data, dirs_exist_ok=True)
    print("Restored from Drive:", drive_manifest)

if not manifest.is_file():
    if DATASET_MODE == "pilot":
        force_flag = "--force" if local_data.exists() else ""
        !musicinstruct generate-pilot --output-dir {LOCAL_DATA} --target {TARGET_ITEMS} {force_flag}
    else:
        lakh = Path(LAKH_ROOT)
        if USE_DRIVE and not lakh.is_dir() and Path(DRIVE_MIDICAPS).is_dir():
            shutil.copytree(DRIVE_MIDICAPS, Path(LAKH_ROOT).parent, dirs_exist_ok=True)
            print("Restored MidiCaps/Lakh from Drive:", DRIVE_MIDICAPS)
        if not lakh.is_dir():
            raise FileNotFoundError(
                f"No Lakh tree at {LAKH_ROOT}. Upload midicaps/lmd_full to Drive or set LAKH_ROOT."
            )
        force_flag = "--force" if local_data.exists() else ""
        !musicinstruct generate-real --lakh-root {LAKH_ROOT} --output-dir {LOCAL_DATA} \
            --target {TARGET_ITEMS} --seed-limit {SEED_LIMIT} \
            --midicaps-limit {MIDICAPS_SCAN_LIMIT} --manifest-name {MANIFEST_NAME} {force_flag}
else:
    print("Using manifest:", manifest)

In [ ]:
# @title Validate & self-test grader
!musicinstruct validate {LOCAL_DATA}/{MANIFEST_NAME}
!musicinstruct self-test {LOCAL_DATA}/{MANIFEST_NAME}

In [ ]:
# @title Dataset summary
import json
from collections import Counter
from pathlib import Path

manifest_path = Path(LOCAL_DATA) / MANIFEST_NAME
rows = [json.loads(line) for line in manifest_path.read_text().splitlines() if line.strip()]
print("mode:", DATASET_MODE)
print("items:", len(rows))
print("splits:", dict(Counter(r["split"] for r in rows)))
test_rows = [r for r in rows if r["split"] == "test"]
print("test items:", len(test_rows))
print("test compositions:", len({r["composition_id"] for r in test_rows}))
print("sample test compositions:", sorted({r["composition_id"] for r in test_rows})[:10])

In [ ]:
# @title Sync data to Google Drive
import shutil
from pathlib import Path

local_data = Path(LOCAL_DATA)
if USE_DRIVE:
    Path(DRIVE_DATA).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_data, DRIVE_DATA, dirs_exist_ok=True)
    print("Synced →", DRIVE_DATA)
    print("files:", len(list(Path(DRIVE_DATA).rglob("*"))))
else:
    print("Skipped Drive sync. Local path:", local_data)

## Next

1. Confirm Drive sync above.
2. Open **`02_run_plan_executor.ipynb`** (same `RUN_ID`, `USE_DRIVE=True`).